# OmniSub2026 — Visual Speech Recognition (VSR)

> **Competition:** [omni-sub](https://www.kaggle.com/competitions/omni-sub) on Kaggle  
> **Task:** Transcribe silent lip-reading videos into English text  
> **Model:** AutoAVSR — LRS3 VSR (WER 19.1%), Imperial College London  

---

### Before running:
1. **Set runtime to T4 GPU** → Runtime → Change runtime type → T4 GPU  
2. **Paste your Kaggle API token** in Step 1 below  
   - Get it from: kaggle.com → profile → Settings → API → Create New Token

Everything else (model weights, language model, competition data) downloads automatically.

## Step 1 — Kaggle API Token

In [ ]:
import os

# ── Paste your Kaggle API token here ──────────────────────────────────────────
# Get it from: kaggle.com → Settings → API → Create New Token
KAGGLE_API_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
# ──────────────────────────────────────────────────────────────────────────────

assert KAGGLE_API_TOKEN != 'PASTE_YOUR_TOKEN_HERE', \
    'ERROR: Please paste your Kaggle API token above before running!'

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
print('Kaggle token set.')

## Step 2 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle CLI
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.0.0'], check=True)

# Clone AutoAVSR repo
if not os.path.exists('/content/AutoAVSR'):
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages',
        '/content/AutoAVSR'], check=True)

# Install all required packages
packages = [
    'hydra-core>=1.3.2', 'opencv-python>=4.5.5.62', 'scipy>=1.3.0',
    'scikit-image>=0.13.0', 'av>=10.0.0', 'six>=1.16.0',
    'mediapipe', 'gdown>=4.7.3',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# ffmpeg for video decoding
subprocess.run(['apt-get', 'install', '-qq', 'ffmpeg'], check=True)

print('All dependencies installed.')

## Step 3 — Download Competition Data from Kaggle

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/omnisub/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Downloading competition data from Kaggle...')
!KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle competitions download -c omni-sub -p "{DATA_DIR}"

print('Extracting...')
!unzip -q "{DATA_DIR}/omni-sub.zip" -d "{DATA_DIR}"

test_count  = len(list((DATA_DIR / 'test').glob('*.mp4')))
train_count = len(list((DATA_DIR / 'train').iterdir()))
print(f'Test videos  : {test_count}')
print(f'Train folders: {train_count}')

## Step 4 — Download Pretrained VSR Model (~955 MB)

In [ ]:
import gdown

MODEL_DIR = Path('/content/AutoAVSR/benchmarks/LRS3/models/LRS3_V_WER19.1')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ZIP = '/tmp/LRS3_V_WER19.1.zip'

if not (MODEL_DIR / 'model.pth').exists():
    print('Downloading VSR model weights (~955MB)...')
    # Direct Google Drive file ID (from AutoAVSR model zoo)
    gdown.download(id='1t8RHhzDTTvOQkLQhmK1LZGnXRRXOXGi6', output=MODEL_ZIP, quiet=False)

    print('Extracting...')
    !unzip -q "{MODEL_ZIP}" -d /tmp/model_extracted
    !cp /tmp/model_extracted/LRS3_V_WER19.1/model.pth "{MODEL_DIR}/model.pth"
    !cp /tmp/model_extracted/LRS3_V_WER19.1/model.json "{MODEL_DIR}/model.json"
else:
    print('Model already present, skipping download.')

print('Model files:')
!ls -lh "{MODEL_DIR}"

## Step 5 — Download Language Model (~191 MB)

In [ ]:
LM_DIR = Path('/content/AutoAVSR/benchmarks/LRS3/language_models/lm_en_subword')
LM_DIR.mkdir(parents=True, exist_ok=True)
LM_ZIP = '/tmp/lm_en_subword.zip'

if not any(LM_DIR.iterdir()):
    print('Downloading language model (~191MB)...')
    # Direct Google Drive file ID (from AutoAVSR model zoo)
    gdown.download(id='1g31HGxJnnOwYl17b70ObFQZ1TSnPvRQv', output=LM_ZIP, quiet=False)

    print('Extracting...')
    !unzip -q "{LM_ZIP}" -d "{LM_DIR.parent}"
else:
    print('Language model already present, skipping download.')

print('Language model files:')
!ls "{LM_DIR}"

## Step 6 — Load Model

In [ ]:
import sys, torch

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# AutoAVSR must be run from its own directory (config paths are relative)
os.chdir('/content/AutoAVSR')
sys.path.insert(0, '/content/AutoAVSR')

from pipelines.pipeline import InferencePipeline

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Write a no-LM fallback config (CTC-only, used if language model fails to load)
NOLM_CONFIG = 'configs/LRS3_V_WER19.1_nolm.ini'
with open(NOLM_CONFIG, 'w') as f:
    f.write('[input]\nmodality=video\nv_fps=25\n\n'
            '[model]\nv_fps=25\n'
            'model_path=benchmarks/LRS3/models/LRS3_V_WER19.1/model.pth\n'
            'model_conf=benchmarks/LRS3/models/LRS3_V_WER19.1/model.json\n'
            'rnnlm=\nrnnlm_conf=\n\n'
            '[decode]\nbeam_size=10\npenalty=0.0\n'
            'maxlenratio=0.0\nminlenratio=0.0\nctc_weight=0.5\nlm_weight=0.0\n')

# Try full model with language model, fall back to CTC-only
try:
    pipeline = InferencePipeline(
        'configs/LRS3_V_WER19.1.ini',
        device=device, detector='mediapipe', face_track=True
    )
    print('Model loaded with language model (best accuracy).')
except Exception as e:
    print(f'LM unavailable ({e})\nFalling back to CTC-only decoding...')
    pipeline = InferencePipeline(
        NOLM_CONFIG,
        device=device, detector='mediapipe', face_track=True
    )
    print('Model loaded (CTC-only).')

## Step 7 — Run Inference on All Test Videos

In [ ]:
import csv
from tqdm.notebook import tqdm

TEST_DIR   = DATA_DIR / 'test'
SAMPLE_CSV = DATA_DIR / 'sample_submission.csv'
OUTPUT_CSV = DATA_DIR / 'submission.csv'

test_paths = []
with open(SAMPLE_CSV) as f:
    for row in csv.DictReader(f):
        test_paths.append(row['path'])

print(f'Running inference on {len(test_paths)} videos on {device}...\n')

results = []
failed  = []

for video_name in tqdm(test_paths):
    video_path = TEST_DIR / video_name

    if not video_path.exists():
        print(f'MISSING: {video_name}')
        results.append({'path': video_name, 'transcription': ''})
        failed.append(video_name)
        continue

    try:
        transcript = pipeline(str(video_path), landmarks_filename=None)
        transcript = transcript.strip().lower()
    except Exception as e:
        print(f'FAILED {video_name}: {e}')
        transcript = ''
        failed.append(video_name)

    results.append({'path': video_name, 'transcription': transcript})
    tqdm.write(f'  {video_name}: {transcript[:100]}')

print(f'\nDone. {len(results)} processed | {len(failed)} failed.')

## Step 8 — Save & Submit

In [ ]:
import pandas as pd
from google.colab import files

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print('Submission preview:')
print(df.to_string())

# Submit directly to Kaggle
!KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle competitions submit \
    -c omni-sub -f "{OUTPUT_CSV}" \
    -m "AutoAVSR LRS3 VSR WER19.1 pretrained"

# Also download as local backup
files.download(str(OUTPUT_CSV))
print('\nCheck your score: https://www.kaggle.com/competitions/omni-sub/submissions')

In [ ]:
!KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle competitions submit -c omni-sub -f "{OUTPUT_CSV}" -m "AutoAVSR LRS3 VSR WER19.1 pretrained"
print('Submitted! Check: https://www.kaggle.com/competitions/omni-sub/submissions')

In [ ]:
# Uncomment and run after getting baseline score
# !git clone https://github.com/YOUR_USERNAME/OmniSub2026 /content/omnisub_code
# !cd /content/omnisub_code && python src/train.py \
#     --data_root /content/omnisub/data/train \
#     --checkpoint /content/AutoAVSR/benchmarks/LRS3/models/LRS3_V_WER19.1/model.pth \
#     --epochs 5 --batch_size 4
print('Fine-tuning is optional — get baseline first.')